In [79]:
import json
import datetime
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report, accuracy_score

warnings.filterwarnings('ignore', category=UserWarning)

In [80]:
def calculate_distance_km(timestamps, speeds):
    """
    Endomondo gives us speed in km/h and timestamps in epoch seconds.
    Distance (km) = Speed (km/h) * (Time_diff_seconds / 3600)
    """
    if not timestamps or not speeds or len(timestamps) < 2:
        return 0.0
        
    total_km = 0.0
    for i in range(1, len(timestamps)):
        time_diff = timestamps[i] - timestamps[i-1]
        if time_diff < 3600: 
            avg_speed_kmh = (speeds[i] + speeds[i-1]) / 2.0
            total_km += avg_speed_kmh * (time_diff / 3600.0)
            
    return total_km

In [81]:
file_path = 'archive/endomondoHR_proper.json'
output_csv = 'endomondo_all_runs.csv'
processed_runs = []
lines_processed = 0

In [82]:
with open(file_path, 'r') as file:
    for line in file:
        lines_processed += 1
        if "'run'" not in line:
            continue
            
        try:
            clean_line = line.replace("'", '"').replace("False", "false").replace("True", "true").replace("None", "null")
            data = json.loads(clean_line)
            
            if data.get('sport') == 'run':
                user_id = data.get('userId')
                timestamps = data.get('timestamp', [])
                speeds = data.get('speed', [])
                
                if timestamps and len(timestamps) > 2 and speeds:
                    start_epoch = timestamps[0]
                    duration_sec = timestamps[-1] - timestamps[0]
                    
                    if duration_sec > 300:
                        distance_km = calculate_distance_km(timestamps, speeds)
                        
                        if 0.5 < distance_km < 50.0:
                            processed_runs.append({
                                'user_id': user_id,
                                'date': datetime.datetime.fromtimestamp(start_epoch).strftime('%Y-%m-%d'),
                                'distance_km': round(distance_km, 2),
                                'duration_sec': duration_sec,
                                'avg_hr': sum(data['heart_rate']) / len(data['heart_rate']) if 'heart_rate' in data else None
                            })
        except Exception:
            pass

In [83]:
df_all = pd.DataFrame(processed_runs)
df_all.to_csv(output_csv, index=False)

In [84]:
# Clean and calculate pace
df_clean = df_all.copy()
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean['pace_min_per_km'] = (df_clean['duration_sec'] / 60) / df_clean['distance_km']
df_clean = df_clean[(df_clean['pace_min_per_km'] >= 3.0) & (df_clean['pace_min_per_km'] <= 15.0)].copy()

In [85]:
def create_features(user_data):
    user_data = user_data.sort_values('date')
    user_data.set_index('date', inplace=True)
    
    # 1. Volume Features
    user_data['dist_last_30d'] = user_data['distance_km'].rolling('30D', closed='left').sum().fillna(0)
    user_data['avg_pace_last_30d'] = user_data['pace_min_per_km'].rolling('30D', closed='left').mean()
    user_data['dist_last_7d'] = user_data['distance_km'].rolling('7D', closed='left').sum().fillna(0)
    
    chronic_load = user_data['dist_last_30d'] / 4.0
    user_data['ACWR'] = user_data['dist_last_7d'] / (chronic_load + 0.1) 
    
    # 2. Behavior & Recovery Features
    user_data['days_since_last_run'] = user_data.index.to_series().diff().dt.days.fillna(7)
    user_data['day_of_week'] = user_data.index.dayofweek
    user_data['runs_last_30d'] = user_data['distance_km'].rolling('30D', closed='left').count().fillna(0)
    user_data['avg_pace_last_7d'] = user_data['pace_min_per_km'].rolling('7D', closed='left').mean()
    
    # 3. Distance Ceiling Features
    user_data['max_dist_last_30d'] = user_data['distance_km'].rolling('30D', closed='left').max().fillna(0)
    user_data['is_weekend'] = user_data['day_of_week'].isin([5, 6]).astype(int)
    
    # 4. Target Variables
    user_data['TARGET_distance'] = user_data['distance_km']
    user_data['TARGET_pace'] = user_data['pace_min_per_km']
    
    return user_data.reset_index()

In [86]:
df_features = df_clean.groupby('user_id', group_keys=False).apply(create_features)
df_final = df_features.dropna(subset=['avg_pace_last_30d', 'avg_pace_last_7d'])
print(f"Data ready for Machine Learning! Final row count: {len(df_final)}")

Data ready for Machine Learning! Final row count: 7048


C:\Users\notmu\AppData\Local\Temp\ipykernel_8140\3935724493.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_features = df_clean.groupby('user_id', group_keys=False).apply(create_features)


In [87]:
features = [
    'dist_last_7d', 'dist_last_30d', 'avg_pace_last_30d', 'ACWR',
    'days_since_last_run', 'day_of_week', 'runs_last_30d', 
    'avg_pace_last_7d', 'max_dist_last_30d', 'is_weekend'
]
X = df_final[features]

In [88]:
# CLASSIFICATION MODEL (Distance Category)
def categorize_distance(dist):
    if dist < 5.0: return 0
    elif dist <= 15.0: return 1
    else: return 2

y_category = df_final['TARGET_distance'].apply(categorize_distance)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_category, test_size=0.2, random_state=42)

clf_model = RandomForestClassifier(n_estimators=100, max_depth=8, class_weight='balanced', random_state=42)
clf_model.fit(X_train_c, y_train_c)
preds_class = clf_model.predict(X_test_c)

print(f"Overall Accuracy: {accuracy_score(y_test_c, preds_class) * 100:.1f}%\n")
print(classification_report(y_test_c, preds_class, target_names=['Short (<5km)', 'Medium (5-15km)', 'Long (>15km)']))

Overall Accuracy: 63.8%

                 precision    recall  f1-score   support

   Short (<5km)       0.39      0.41      0.40       101
Medium (5-15km)       0.78      0.68      0.73       955
   Long (>15km)       0.44      0.60      0.51       354

       accuracy                           0.64      1410
      macro avg       0.54      0.56      0.54      1410
   weighted avg       0.67      0.64      0.65      1410



In [89]:
# REGRESSION MODEL (Exact Pace)

y_reg = df_final[['TARGET_distance', 'TARGET_pace']]
X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

rmse_pace = np.sqrt(mean_squared_error(y_test['TARGET_pace'], predictions[:, 1]))
r2_pace = r2_score(y_test['TARGET_pace'], predictions[:, 1])

print(f"R-squared: {r2_pace:.2f} (1.0 is perfect)")
print(f"RMSE:      {rmse_pace:.2f} min/km")

R-squared: 0.57 (1.0 is perfect)
RMSE:      0.57 min/km


In [90]:
joblib.dump(clf_model, 'strideup_distance_classifier.pkl')
joblib.dump(model, 'strideup_pace_regressor.pkl')

['strideup_pace_regressor.pkl']